# COMPASS preprocessing

Stage 0-3 of the COMPASS survival pipeline: schema audit (profile_data only),
cohort compile, longitudinal lab preprocessing, prediction-input build, and
cohort diagnostics. Univariate/multivariate modeling live in
`02_univariate.ipynb` / `03_multivariate.ipynb` and only read the
`prediction_inputs_<arm>/` files this notebook writes. All stages use the
merged `profile_data` parquets.

In [ ]:
ARMS = ["adt"]
ENDPOINTS = ("platinum", "nepc", "avpc", "avpc_nepc")
COHORTS = ("all", "metastatic", "localized")
# Stage 3 writes one independent tree per cohort x endpoint. Each build applies
# only its own time-validity gate: t_platinum never filters NEPC/AVPC and
# t_nepc/t_avpc never filter platinum. The metastatic/localized cohorts are the
# medication-derived ADT-intent strata, applied as an MRN restriction at Stage 3
# (see Stage 1b); "all" is the unrestricted ADT cohort. Stages 0-2 are shared and
# run once per treatment anchor.

import sys
sys.path.insert(0, ".")
import compass_pipeline as cp

RUNS = cp.make_endpoint_runs(ARMS, endpoints=ENDPOINTS, cohorts=COHORTS)


## Stage 0 -- schema audit

Fails fast if a required column is absent or all-null in the merged sources.

In [ ]:
cp.audit_schema()

## Stage 1 -- compile COMPASS cohort data

Endpoint-independent: writes one survival cohort carrying **all four**
endpoints' columns (`PLATINUM`/`TT_PLATINUM`, `NEPC`/`TT_NEPC`,
`AVPC`/`TT_AVPC`, `AVPC_NEPC`/`TT_AVPC_NEPC`). The NEPC, AVPC, and AVPC_NEPC
columns all come from the same broader LLM criteria-timeline labels
(`LLM_annotations/LLM_avpc_nepc_timeline/avpc_nepc_labels.parquet`): `nepc` is
the timeline's NEPC-only component, `avpc` is its AVPC-only component (>=3
Aparicio criteria), and `avpc_nepc` is their union. If that file is not
mounted the stage still succeeds and simply emits no NEPC/AVPC/AVPC_NEPC
columns, leaving the platinum pipeline unaffected.

Read the printed summary before spending modelling effort: it reports each
endpoint's positive count, the `date_source` / `date_precision` / `label_source`
breakdowns, and how many events are **prevalent** (at or before the anchor)
and will therefore be excluded by the incident-endpoint filter.

In [ ]:
cp.compile_cohort(arms=ARMS)

## Stage 1b -- medication-derived ADT-intent strata

Writes a combined audit label, one MRN list per stratum, and preliminary
incident endpoint counts under `<data_root>/mrn_lists/`. Stage 3 consumes those
MRN lists via `--restrict-to-mrns` to build the `metastatic` and `localized`
cohort trees; the count table is an early warning for cohort/endpoint pairs with
too few events to fit reliably.

**Interpretation warning:** `ADT_INTENT` uses the full observed ADT course
(duration, cessation/restart, and later definitive escalation). Platinum is
excluded from the classifier, but the stratum is still not available
prospectively at the landmark. Cohort-stratified results are retrospective.


In [ ]:
STRATUM_FILES = cp.build_adt_intent_mrn_lists()

import pandas as pd

preliminary_counts = pd.read_csv(STRATUM_FILES["counts"])
preliminary_counts


## Stage 2 -- preprocess raw labs (per arm anchor)

Expensive: full raw lab standardization. The Parquet cache
(`consolidated_longitudinal_data_<arm>.parquet`) makes reruns cheap, but the
first pass may be slow.

In [ ]:
for run in cp.stage2_runs(RUNS):
    cp.preprocess_labs(run)


## Stage 3 -- build prediction inputs + cohort diagnostics

Set `REBUILD_PREDICTION_INPUTS = False` to skip rebuilding. This cell builds both endpoint trees. Platinum requires only valid `t_platinum`; NEPC requires only valid incident `t_nepc`. Shared death/follow-up checks still apply to both.

In [ ]:
REBUILD_PREDICTION_INPUTS = True

for run in RUNS:
    if REBUILD_PREDICTION_INPUTS:
        cp.build_prediction_inputs(run)
    else:
        print(f"[skip] prediction-input rebuild disabled for {run['label']}")
    cp.cohort_diagnostics(run)

## Stage 3b -- sequencing, Gleason, and PRS inputs

Builds `prediction_inputs_<arm>/somatic_gleason/` from the sample-level
somatic matrix published by `PROFILE_data_processing` and the Gleason timeline
published by `LLM_clinical_annotations`. It creates two cohorts: the sequencing
sample closest to ADT start with follow-up from specimen collection, and the
Gleason score closest to ADT start with follow-up from the score date. PRSs use
ADT start itself as the prediction origin.

In [ ]:
for run in RUNS:
    cp.build_somatic_gleason_inputs(run)